In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ============================================================================
# PHASE 0: LOAD VIBETHINKER-3B MODEL
# Project: Test-Time Verification for Long-Horizon Math Reasoning
# ============================================================================

import os
import sys
import torch
import json
from datetime import datetime

# Check if GPU is available
print("=" * 80)
print("PHASE 0: ENVIRONMENT CHECK")
print("=" * 80)
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Type: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print()

# Create output directory for results
os.makedirs("outputs", exist_ok=True)
print(f"✅ Output directory created at: /kaggle/working/outputs")
print()

# ============================================================================
# STEP 1: LOAD VIBETHINKER-3B MODEL
# ============================================================================
print("=" * 80)
print("LOADING VIBETHINKER-3B MODEL (frozen, read-only)")
print("=" * 80)
print("This will download the model (~6 GB) from Hugging Face...")
print("(This takes 2-3 minutes; be patient!)")
print()

# Load the model and tokenizer
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    
    model_name = "WeiboAI/VibeThinker-3B"
    print(f"Model: {model_name}")
    print("Downloading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print("✅ Tokenizer loaded")
    
    print("Downloading model weights (this may take a few minutes)...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,  # Use half precision to save memory
        device_map="auto"           # Automatically put model on GPU
    )
    print("✅ Model loaded onto GPU")
    print()
    
    # Verify model is on GPU
    print(f"Model device: {next(model.parameters()).device}")
    print(f"Model dtype: {next(model.parameters()).dtype}")
    print()
    
except Exception as e:
    print(f"❌ ERROR loading model: {e}")
    print("This might mean:")
    print("  - GPU ran out of memory (restart session and try again)")
    print("  - Internet connection dropped (check your connection)")
    sys.exit(1)

print("=" * 80)
print("✅ PHASE 0 COMPLETE: Model loaded successfully!")
print("=" * 80)
print()
print("What's next:")
print("  - Model is frozen (we won't update it)")
print("  - Model is on GPU (inference will be fast)")
print("  - Ready for Phase 1: Baseline evaluation")
print()

# Save status to a log file
status_log = {
    "timestamp": datetime.now().isoformat(),
    "phase": "Phase 0",
    "status": "SUCCESS",
    "model": model_name,
    "gpu_available": torch.cuda.is_available(),
    "gpu_type": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
}

with open("outputs/phase0_status.json", "w") as f:
    json.dump(status_log, f, indent=2)

print("✅ Status saved to: outputs/phase0_status.json")

In [ ]:
# ============================================================================
# PHASE 1: SPEED TEST
# Goal: measure real tokens/second so we can budget the whole project
# ============================================================================

import time, json, torch

print("=" * 80)
print("PHASE 1: SPEED TEST")
print("=" * 80)
print()


def run_test(name, question, max_new_tokens):
    """Ask the model one question and measure how fast it generates."""
    print("-" * 80)
    print(f"TEST: {name}")
    print("-" * 80)
    print(f"Question: {question[:100]}...")
    print(f"Generating (limit {max_new_tokens} tokens)... please wait")
    print()

    # Format the question the way VibeThinker expects
    messages = [{"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    n_input = inputs.input_ids.shape[1]

    # Generate, with a stopwatch running
    torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    elapsed = time.time() - start

    n_generated = out.shape[1] - n_input
    tok_per_sec = n_generated / elapsed
    answer = tokenizer.decode(out[0][n_input:], skip_special_tokens=True)
    finished = n_generated < max_new_tokens   # did it stop on its own?

    print(f"  Time taken      : {elapsed:.1f} seconds")
    print(f"  Tokens generated: {n_generated}")
    print(f"  SPEED           : {tok_per_sec:.1f} tokens/second")
    print(f"  Finished early? : {'YES (natural stop)' if finished else 'NO (hit the limit)'}")
    print()
    print("  --- First 400 characters of the answer ---")
    print("  " + answer[:400].replace("\n", "\n  "))
    print("  ---")
    print()

    return {
        "name": name,
        "seconds": round(elapsed, 1),
        "tokens_generated": n_generated,
        "tokens_per_second": round(tok_per_sec, 1),
        "finished_naturally": finished,
    }


results = []

# Test 1: something trivial, just to confirm generation works at all
results.append(run_test(
    "Warm-up (easy question)",
    "What is 17 multiplied by 23? Show your reasoning.",
    max_new_tokens=512,
))

# Test 2: a real AIME-difficulty problem - this is the realistic speed
results.append(run_test(
    "Real AIME-style problem",
    "Find the number of ordered pairs of positive integers (a, b) such that "
    "a + b = 1000 and neither a nor b has a zero digit.",
    max_new_tokens=3000,
))

# ---------------------------------------------------------------------------
# Budget projection
# ---------------------------------------------------------------------------
speed = results[-1]["tokens_per_second"]     # use the realistic test
AVG_TOKENS_PER_ANSWER = 6000                 # typical for a reasoning model

sec_per_answer = AVG_TOKENS_PER_ANSWER / speed

print("=" * 80)
print("PROJECT BUDGET PROJECTION")
print("=" * 80)
print(f"Measured speed          : {speed:.1f} tokens/sec")
print(f"Est. time for 1 answer  : {sec_per_answer/60:.1f} minutes")
print()
print("If we run 30 questions with K samples each:")
for K in [1, 4, 8, 16, 32]:
    hours = (30 * K * sec_per_answer) / 3600
    verdict = "OK" if hours < 8 else ("TIGHT" if hours < 25 else "TOO SLOW")
    print(f"  K={K:2d}  ->  {hours:7.1f} GPU-hours   [{verdict}]")
print()
print("(Our total weekly budget is 30 GPU-hours)")
print("=" * 80)

with open("outputs/phase1_speed.json", "w") as f:
    json.dump({"tests": results, "tokens_per_second": speed}, f, indent=2)
print("Saved to: outputs/phase1_speed.json")

In [ ]:
# ============================================================================
# PHASE 2A: INSTALL vLLM
# This takes 5-10 minutes. It downloads a lot. That is normal.
# ============================================================================

!pip install -q vllm

print()
print("=" * 80)
print("INSTALL FINISHED")
print("=" * 80)
print()
print(">>> NOW YOU MUST RESTART THE KERNEL. See instructions below. <<<")

In [ ]:
# ============================================================================
# PHASE 2B: LOAD WITH vLLM + MEASURE REAL SPEED
# ============================================================================

import time, json, os, torch

os.makedirs("outputs", exist_ok=True)

# --- Fast compatibility check (fails in seconds, not minutes) ---------------
print("=" * 80)
print("COMPATIBILITY CHECK")
print("=" * 80)

import vllm
print(f"vLLM version   : {vllm.__version__}")
print(f"PyTorch version: {torch.__version__}")

cap = torch.cuda.get_device_capability(0)
print(f"GPU            : {torch.cuda.get_device_name(0)}")
print(f"Compute capab. : {cap[0]}.{cap[1]}   (T4 = 7.5 = Turing)")
if cap[0] < 8:
    print("NOTE: Turing GPU. Must use float16. Some vLLM backends unavailable.")
print()

# --- Load the model ---------------------------------------------------------
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

MODEL = "WeiboAI/VibeThinker-3B"

print("=" * 80)
print("LOADING MODEL WITH vLLM  (2-4 min, log spam is normal)")
print("=" * 80)

tokenizer = AutoTokenizer.from_pretrained(MODEL)

llm = LLM(
    model=MODEL,
    dtype="float16",              # Turing cannot do bfloat16
    gpu_memory_utilization=0.90,
    max_model_len=8192,
    trust_remote_code=True,
)

print("\nModel loaded.\n")


def build_prompt(question):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        tokenize=False,
        add_generation_prompt=True,
    )


QUESTION = (
    "Find the number of ordered pairs of positive integers (a, b) such that "
    "a + b = 1000 and neither a nor b has a zero digit."
)

results = {}

for K in [1, 8, 32]:
    print("-" * 80)
    print(f"TEST: K={K} samples generated together")
    print("-" * 80)

    params = SamplingParams(
        n=K,                      # K samples from ONE prompt, batched
        temperature=1.0,
        top_p=0.95,
        max_tokens=6000,
    )

    start = time.time()
    outputs = llm.generate([build_prompt(QUESTION)], params)
    elapsed = time.time() - start

    comps = outputs[0].outputs
    lengths = [len(c.token_ids) for c in comps]
    total = sum(lengths)
    n_done = sum(1 for c in comps if c.finish_reason == "stop")

    print(f"  Wall-clock time    : {elapsed:.1f} sec")
    print(f"  Total tokens       : {total:,}")
    print(f"  THROUGHPUT         : {total/elapsed:.1f} tokens/sec")
    print(f"  Avg length         : {total//len(lengths)} tokens")
    print(f"  Longest            : {max(lengths)} tokens")
    print(f"  Finished naturally : {n_done}/{len(comps)}")
    print()

    results[f"K={K}"] = {
        "seconds": round(elapsed, 1),
        "throughput": round(total / elapsed, 1),
        "avg_len": total // len(lengths),
        "max_len": max(lengths),
        "finished_naturally": f"{n_done}/{len(comps)}",
    }

# --- Re-project the budget with real numbers --------------------------------
sec32 = results["K=32"]["seconds"]
OLD_HOURS = 105.3

print("=" * 80)
print("NEW BUDGET PROJECTION (30 questions per benchmark)")
print("=" * 80)
print(f"Time per question at K=32 : {sec32/60:.1f} min")
print()
for K in [4, 8, 16, 32]:
    hrs = (30 * sec32 * (K / 32)) / 3600
    verdict = "OK" if hrs < 8 else ("TIGHT" if hrs < 20 else "TOO SLOW")
    print(f"  K={K:2d}  ->  {hrs:6.2f} GPU-hours   [{verdict}]")

new_hours = (30 * sec32) / 3600
print()
print(f"Speedup vs HuggingFace: {OLD_HOURS/new_hours:.1f}x")
print("=" * 80)

with open("outputs/phase2_vllm_speed.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved to: outputs/phase2_vllm_speed.json")

In [ ]:
# ============================================================================
# PHASE 3A: LOCATE A REAL MATH BENCHMARK DATASET
# We probe several known dataset IDs and report which ones actually load.
# ============================================================================

from datasets import load_dataset
import json, os

os.makedirs("outputs", exist_ok=True)

CANDIDATES = [
    "HuggingFaceH4/aime_2024",
    "math-ai/aime25",
    "opencompass/AIME2025",
    "yentinglin/aime_2025",
    "AI-MO/aimo-validation-aime",
    "Maxwell-Jia/AIME_2024",
]

print("=" * 80)
print("PROBING DATASETS")
print("=" * 80)

available = {}

for name in CANDIDATES:
    try:
        ds = load_dataset(name)
        split = list(ds.keys())[0]
        n = len(ds[split])
        cols = ds[split].column_names
        print(f"  OK    {name}")
        print(f"        split='{split}'  rows={n}")
        print(f"        columns={cols}")
        available[name] = {"split": split, "rows": n, "columns": cols}
        # show one example so we can see the format
        ex = ds[split][0]
        for c in cols:
            v = str(ex[c])
            print(f"        {c}: {v[:150]}{'...' if len(v) > 150 else ''}")
        print()
    except Exception as e:
        print(f"  FAIL  {name}")
        print(f"        {str(e)[:120]}")
        print()

print("=" * 80)
print(f"USABLE DATASETS FOUND: {len(available)}")
for k in available:
    print(f"  - {k}")
print("=" * 80)

with open("outputs/phase3_datasets.json", "w") as f:
    json.dump(available, f, indent=2)
print("Saved to: outputs/phase3_datasets.json")

In [ ]:
# ============================================================================
# PHASE 3B: REAL AIME PROBLEMS - HOW LONG DOES THE MODEL ACTUALLY THINK?
# Also gives us our FIRST real accuracy signal.
# ============================================================================

import time, json, os, re
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset

os.makedirs("outputs", exist_ok=True)
MODEL = "WeiboAI/VibeThinker-3B"

# --- Load 4 real AIME 2025 problems ----------------------------------------
ds = load_dataset("math-ai/aime25")["test"]
problems = [ds[i] for i in range(4)]

print("=" * 80)
print("TEST PROBLEMS (real AIME 2025)")
print("=" * 80)
for p in problems:
    print(f"  [{p['id']}] answer={p['answer']}  |  {p['problem'][:80]}...")
print()

# --- Load model with a MUCH bigger context window ---------------------------
print("Loading vLLM with max_model_len=16384 (may recompile, be patient)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
llm = LLM(
    model=MODEL,
    dtype="float16",
    gpu_memory_utilization=0.90,
    max_model_len=16384,          # doubled
    trust_remote_code=True,
)
print("Loaded.\n")


def build_prompt(q):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}],
        tokenize=False, add_generation_prompt=True,
    )


def extract_boxed(text):
    """Pull the final \\boxed{...} value out of a reasoning trace."""
    idx = text.rfind("\\boxed{")
    if idx == -1:
        return None
    i, depth, out = idx + 7, 1, []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
        out.append(c)
        i += 1
    return "".join(out).strip()


def to_int(s):
    """AIME answers are integers 0-999."""
    if s is None:
        return None
    m = re.search(r"-?\d+", s.replace(",", ""))
    return int(m.group()) if m else None


# --- Generate: 4 problems x 4 samples, with a HIGH ceiling ------------------
K = 4
MAX_TOK = 15000

params = SamplingParams(n=K, temperature=1.0, top_p=0.95, max_tokens=MAX_TOK)
prompts = [build_prompt(p["problem"]) for p in problems]

print("=" * 80)
print(f"GENERATING  {len(prompts)} problems x K={K}  (cap {MAX_TOK} tokens)")
print("This is the long one. Expect 15-30 minutes.")
print("=" * 80)

start = time.time()
outputs = llm.generate(prompts, params)
elapsed = time.time() - start

# --- Analyse ----------------------------------------------------------------
all_lengths, n_finished, n_total = [], 0, 0
rows, correct_any = [], 0

print()
print("=" * 80)
print("RESULTS PER PROBLEM")
print("=" * 80)

for p, out in zip(problems, outputs):
    truth = to_int(str(p["answer"]))
    lens, preds, fin = [], [], 0
    for c in out.outputs:
        L = len(c.token_ids)
        lens.append(L)
        all_lengths.append(L)
        n_total += 1
        if c.finish_reason == "stop":
            fin += 1
            n_finished += 1
        preds.append(to_int(extract_boxed(c.text)))

    hits = sum(1 for x in preds if x is not None and x == truth)
    parsed = sum(1 for x in preds if x is not None)
    if hits > 0:
        correct_any += 1

    print(f"[{p['id']}] truth={truth}")
    print(f"    lengths        : {lens}")
    print(f"    finished       : {fin}/{K}")
    print(f"    answers parsed : {parsed}/{K}   predictions={preds}")
    print(f"    correct        : {hits}/{K}")
    print()

    rows.append({
        "id": p["id"], "truth": truth, "lengths": lens,
        "finished": fin, "parsed": parsed,
        "predictions": preds, "correct": hits,
    })

all_lengths.sort()
med = all_lengths[len(all_lengths) // 2]
p90 = all_lengths[int(len(all_lengths) * 0.9)]

print("=" * 80)
print("LENGTH DISTRIBUTION  (the number we came for)")
print("=" * 80)
print(f"  Finished naturally : {n_finished}/{n_total}  ({100*n_finished/n_total:.0f}%)")
print(f"  Shortest           : {all_lengths[0]:,} tokens")
print(f"  Median             : {med:,} tokens")
print(f"  90th percentile    : {p90:,} tokens")
print(f"  Longest            : {all_lengths[-1]:,} tokens")
print()
print(f"  Wall-clock         : {elapsed/60:.1f} min for {n_total} generations")
print(f"  Throughput         : {sum(all_lengths)/elapsed:.1f} tokens/sec")
print()
print("=" * 80)
print("EARLY ACCURACY SIGNAL")
print("=" * 80)
print(f"  Problems with >=1 correct sample : {correct_any}/{len(problems)}")
tot_hits = sum(r['correct'] for r in rows)
print(f"  Total correct samples            : {tot_hits}/{n_total}  ({100*tot_hits/n_total:.0f}%)")
print("  (Paper reports 91.4 on AIME25 - small sample, so expect noise)")
print("=" * 80)

with open("outputs/phase3_lengths.json", "w") as f:
    json.dump({
        "per_problem": rows,
        "finished_rate": n_finished / n_total,
        "median_len": med, "p90_len": p90, "max_len": all_lengths[-1],
        "throughput": round(sum(all_lengths) / elapsed, 1),
        "minutes": round(elapsed / 60, 1),
    }, f, indent=2)
print("Saved to: outputs/phase3_lengths.json")

In [ ]:
# ============================================================================
# CELL 0 - SESSION SETUP
# Run this FIRST every time you open the notebook after a break.
# It is smart: if vLLM is already there, it skips the install instantly.
# ============================================================================

import importlib.util, os

os.makedirs("outputs", exist_ok=True)

if importlib.util.find_spec("vllm") is None:
    print("vLLM not found (fresh Kaggle session).")
    print("Installing now - takes 5-10 minutes. Ignore the red conflict warnings.")
    print()
    !pip install -q vllm
    print()
    print("=" * 60)
    print("INSTALL DONE.")
    print("=" * 60)
else:
    import vllm
    print(f"vLLM already installed (version {vllm.__version__}). Nothing to do.")

print()
print("Setup complete. You can run the next cell.")

In [ ]:
# ============================================================================
# PHASE 3B: REAL AIME PROBLEMS - HOW LONG DOES THE MODEL ACTUALLY THINK?
# Also gives us our FIRST real accuracy signal.
# ============================================================================

import time, json, os, re
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset

os.makedirs("outputs", exist_ok=True)
MODEL = "WeiboAI/VibeThinker-3B"

# --- Load 4 real AIME 2025 problems ----------------------------------------
ds = load_dataset("math-ai/aime25")["test"]
problems = [ds[i] for i in range(4)]

print("=" * 80)
print("TEST PROBLEMS (real AIME 2025)")
print("=" * 80)
for p in problems:
    print(f"  [{p['id']}] answer={p['answer']}  |  {p['problem'][:80]}...")
print()

# --- Load model with a MUCH bigger context window ---------------------------
print("Loading vLLM with max_model_len=16384 (may recompile, be patient)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
llm = LLM(
    model=MODEL,
    dtype="float16",
    gpu_memory_utilization=0.90,
    max_model_len=16384,          # doubled
    trust_remote_code=True,
)
print("Loaded.\n")


def build_prompt(q):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}],
        tokenize=False, add_generation_prompt=True,
    )


def extract_boxed(text):
    """Pull the final \\boxed{...} value out of a reasoning trace."""
    idx = text.rfind("\\boxed{")
    if idx == -1:
        return None
    i, depth, out = idx + 7, 1, []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
        out.append(c)
        i += 1
    return "".join(out).strip()


def to_int(s):
    """AIME answers are integers 0-999."""
    if s is None:
        return None
    m = re.search(r"-?\d+", s.replace(",", ""))
    return int(m.group()) if m else None


# --- Generate: 4 problems x 4 samples, with a HIGH ceiling ------------------
K = 4
MAX_TOK = 15000

params = SamplingParams(n=K, temperature=1.0, top_p=0.95, max_tokens=MAX_TOK)
prompts = [build_prompt(p["problem"]) for p in problems]

print("=" * 80)
print(f"GENERATING  {len(prompts)} problems x K={K}  (cap {MAX_TOK} tokens)")
print("This is the long one. Expect 15-30 minutes.")
print("=" * 80)

start = time.time()
outputs = llm.generate(prompts, params)
elapsed = time.time() - start

# --- Analyse ----------------------------------------------------------------
all_lengths, n_finished, n_total = [], 0, 0
rows, correct_any = [], 0

print()
print("=" * 80)
print("RESULTS PER PROBLEM")
print("=" * 80)

for p, out in zip(problems, outputs):
    truth = to_int(str(p["answer"]))
    lens, preds, fin = [], [], 0
    for c in out.outputs:
        L = len(c.token_ids)
        lens.append(L)
        all_lengths.append(L)
        n_total += 1
        if c.finish_reason == "stop":
            fin += 1
            n_finished += 1
        preds.append(to_int(extract_boxed(c.text)))

    hits = sum(1 for x in preds if x is not None and x == truth)
    parsed = sum(1 for x in preds if x is not None)
    if hits > 0:
        correct_any += 1

    print(f"[{p['id']}] truth={truth}")
    print(f"    lengths        : {lens}")
    print(f"    finished       : {fin}/{K}")
    print(f"    answers parsed : {parsed}/{K}   predictions={preds}")
    print(f"    correct        : {hits}/{K}")
    print()

    rows.append({
        "id": p["id"], "truth": truth, "lengths": lens,
        "finished": fin, "parsed": parsed,
        "predictions": preds, "correct": hits,
    })

all_lengths.sort()
med = all_lengths[len(all_lengths) // 2]
p90 = all_lengths[int(len(all_lengths) * 0.9)]

print("=" * 80)
print("LENGTH DISTRIBUTION  (the number we came for)")
print("=" * 80)
print(f"  Finished naturally : {n_finished}/{n_total}  ({100*n_finished/n_total:.0f}%)")
print(f"  Shortest           : {all_lengths[0]:,} tokens")
print(f"  Median             : {med:,} tokens")
print(f"  90th percentile    : {p90:,} tokens")
print(f"  Longest            : {all_lengths[-1]:,} tokens")
print()
print(f"  Wall-clock         : {elapsed/60:.1f} min for {n_total} generations")
print(f"  Throughput         : {sum(all_lengths)/elapsed:.1f} tokens/sec")
print()
print("=" * 80)
print("EARLY ACCURACY SIGNAL")
print("=" * 80)
print(f"  Problems with >=1 correct sample : {correct_any}/{len(problems)}")
tot_hits = sum(r['correct'] for r in rows)
print(f"  Total correct samples            : {tot_hits}/{n_total}  ({100*tot_hits/n_total:.0f}%)")
print("  (Paper reports 91.4 on AIME25 - small sample, so expect noise)")
print("=" * 80)

with open("outputs/phase3_lengths.json", "w") as f:
    json.dump({
        "per_problem": rows,
        "finished_rate": n_finished / n_total,
        "median_len": med, "p90_len": p90, "max_len": all_lengths[-1],
        "throughput": round(sum(all_lengths) / elapsed, 1),
        "minutes": round(elapsed / 60, 1),
    }, f, indent=2)
print("Saved to: outputs/phase3_lengths.json")

In [ ]:
# ============================================================================
# PHASE 4: WHY DOES PROBLEM 1 NEVER FINISH?
# Repetition loop, or genuinely needs more room?
# ============================================================================

import time, json
from vllm import SamplingParams

prob = ds[1]                      # the problem that never finished
print("=" * 80)
print("PROBLEM UNDER TEST")
print("=" * 80)
print(f"id     : {prob['id']}")
print(f"answer : {prob['answer']}")
print(f"text   : {prob['problem']}")
print()

K, MAX_TOK = 8, 24000
print(f"Running K={K} samples with a {MAX_TOK:,} token ceiling.")
print("Expect roughly 20-30 minutes.")
print()

params = SamplingParams(n=K, temperature=1.0, top_p=0.95, max_tokens=MAX_TOK)

start = time.time()
out = llm.generate([build_prompt(prob["problem"])], params)[0]
elapsed = time.time() - start


def repetition_score(token_ids, tail=100, chunk=25):
    """Fraction of tail chunks that already appeared earlier. High = looping."""
    if len(token_ids) < tail * 3:
        return 0.0
    tail_ids = list(token_ids[-tail:])
    body = list(token_ids[:-tail])
    body_str = ",".join(map(str, body))
    hits = 0
    n_chunks = tail // chunk
    for i in range(n_chunks):
        piece = ",".join(map(str, tail_ids[i * chunk:(i + 1) * chunk]))
        if piece and piece in body_str:
            hits += 1
    return hits / n_chunks


truth = to_int(str(prob["answer"]))
print("=" * 80)
print("PER-SAMPLE RESULTS")
print("=" * 80)

n_fin, rows = 0, []
for i, c in enumerate(out.outputs):
    L = len(c.token_ids)
    fin = c.finish_reason == "stop"
    n_fin += fin
    pred = to_int(extract_boxed(c.text))
    rep = repetition_score(c.token_ids)
    print(f"  sample {i}: len={L:6,}  finished={str(fin):5}  "
          f"pred={pred}  repetition={rep:.0%}")
    rows.append({"len": L, "finished": fin, "pred": pred, "repetition": rep})

print()
print(f"  Finished naturally : {n_fin}/{K}")
print(f"  Wall-clock         : {elapsed/60:.1f} min")
print(f"  Throughput         : {sum(len(c.token_ids) for c in out.outputs)/elapsed:.1f} tok/s")
print()

# --- Show the actual text so we can see with our own eyes -------------------
worst = max(out.outputs, key=lambda c: len(c.token_ids))
print("=" * 80)
print("LAST 1200 CHARACTERS OF THE LONGEST SAMPLE")
print("(If you see the same sentences repeating -> it is a loop)")
print("=" * 80)
print(worst.text[-1200:])
print()

print("=" * 80)
print("MIDDLE SECTION (around the 50% mark) FOR COMPARISON")
print("=" * 80)
mid = len(worst.text) // 2
print(worst.text[mid:mid + 800])
print()

avg_rep = sum(r["repetition"] for r in rows) / len(rows)
print("=" * 80)
print("VERDICT")
print("=" * 80)
print(f"  Average repetition score: {avg_rep:.0%}")
if n_fin >= K * 0.6:
    print("  -> NEEDS MORE ROOM. 24k is enough. Raise the ceiling.")
elif avg_rep > 0.4:
    print("  -> REPETITION LOOP. More tokens will NOT help. Cap it low.")
else:
    print("  -> INCONCLUSIVE. Read the text above and tell me what you see.")
print("=" * 80)

with open("outputs/phase4_diagnosis.json", "w") as f:
    json.dump({
        "problem_id": prob["id"], "truth": truth,
        "samples": rows, "finished": f"{n_fin}/{K}",
        "avg_repetition": round(avg_rep, 3),
        "minutes": round(elapsed / 60, 1),
    }, f, indent=2)
print("Saved to: outputs/phase4_diagnosis.json")

In [ ]:
# ============================================================================
# PHASE 2A: INSTALL vLLM
# This takes 5-10 minutes. It downloads a lot. That is normal.
# ============================================================================

!pip install -q vllm

print()
print("=" * 80)
print("INSTALL FINISHED")
print("=" * 80)
print()
print(">>> NOW YOU MUST RESTART THE KERNEL. See instructions below. <<<")

In [ ]:
# ============================================================================
# CELL 0 - SESSION SETUP
# Run this FIRST every time you open the notebook after a break.
# It is smart: if vLLM is already there, it skips the install instantly.
# ============================================================================

import importlib.util, os

os.makedirs("outputs", exist_ok=True)

if importlib.util.find_spec("vllm") is None:
    print("vLLM not found (fresh Kaggle session).")
    print("Installing now - takes 5-10 minutes. Ignore the red conflict warnings.")
    print()
    !pip install -q vllm
    print()
    print("=" * 60)
    print("INSTALL DONE.")
    print("=" * 60)
else:
    import vllm
    print(f"vLLM already installed (version {vllm.__version__}). Nothing to do.")

print()
print("Setup complete. You can run the next cell.")

In [ ]:
# ============================================================================
# PHASE 5: FULL AIME25 BASELINE SURVEY
# 30 problems x K=4, big context. Saves progress every 10 problems.
# ============================================================================

import time, json, os, re
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset

os.makedirs("outputs", exist_ok=True)
MODEL = "WeiboAI/VibeThinker-3B"

ds = load_dataset("math-ai/aime25")["test"]
print(f"Loaded {len(ds)} AIME 2025 problems.\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL)

# Try a big context; fall back if the GPU can't hold it
llm = None
for ctx in [32768, 24576, 20480]:
    try:
        print(f"Trying max_model_len={ctx} ...")
        llm = LLM(model=MODEL, dtype="float16", gpu_memory_utilization=0.90,
                  max_model_len=ctx, trust_remote_code=True)
        print(f"SUCCESS with max_model_len={ctx}\n")
        MAX_CTX = ctx
        break
    except Exception as e:
        print(f"  failed: {str(e)[:150]}\n")
if llm is None:
    raise RuntimeError("Could not load model at any context size.")


def build_prompt(q):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)


def extract_boxed(text):
    idx = text.rfind("\\boxed{")
    if idx == -1:
        return None
    i, depth, out = idx + 7, 1, []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
        out.append(c); i += 1
    return "".join(out).strip()


def to_int(s):
    if s is None:
        return None
    m = re.search(r"-?\d+", s.replace(",", ""))
    return int(m.group()) if m else None


K = 4
MAX_TOK = MAX_CTX - 512
params = SamplingParams(n=K, temperature=1.0, top_p=0.95, max_tokens=MAX_TOK)

print("=" * 80)
print(f"RUNNING 30 problems x K={K}, ceiling {MAX_TOK:,} tokens")
print("Processing in 3 chunks of 10 so progress is saved along the way.")
print("Expect 1.5 - 3 hours. Let it run.")
print("=" * 80)
print()

all_rows = []
t0 = time.time()

for chunk_start in range(0, 30, 10):
    idxs = list(range(chunk_start, min(chunk_start + 10, 30)))
    probs = [ds[i] for i in idxs]
    print(f"--- Chunk: problems {idxs[0]}-{idxs[-1]} ---")

    c0 = time.time()
    outs = llm.generate([build_prompt(p["problem"]) for p in probs], params)
    print(f"    chunk time: {(time.time()-c0)/60:.1f} min")

    for p, out in zip(probs, outs):
        truth = to_int(str(p["answer"]))
        lens, preds, fin = [], [], 0
        for c in out.outputs:
            lens.append(len(c.token_ids))
            fin += (c.finish_reason == "stop")
            preds.append(to_int(extract_boxed(c.text)))

        valid = [x for x in preds if x is not None]
        hits = sum(1 for x in valid if x == truth)
        unique = len(set(valid))

        all_rows.append({
            "id": int(p["id"]), "truth": truth, "lengths": lens,
            "finished": fin, "parsed": len(valid), "predictions": preds,
            "correct": hits, "unique_answers": unique,
            "pass_at_1": hits / K,
        })
        flag = "DISAGREE" if unique > 1 else ("TRUNC" if len(valid) == 0 else "")
        print(f"    [{p['id']:2}] truth={truth:4}  correct={hits}/{K}  "
              f"fin={fin}/{K}  maxlen={max(lens):6,}  {flag}")

    with open("outputs/phase5_baseline.json", "w") as f:
        json.dump(all_rows, f, indent=2)
    print(f"    saved ({len(all_rows)}/30 done)\n")

elapsed = time.time() - t0

# ---------------------------------------------------------------------------
n = len(all_rows)
total_correct = sum(r["correct"] for r in all_rows)
total_samples = n * K
solved_all = sum(1 for r in all_rows if r["correct"] == K)
solved_some = sum(1 for r in all_rows if 0 < r["correct"] < K)
solved_none = sum(1 for r in all_rows if r["correct"] == 0)
truncated = sum(1 for r in all_rows if r["parsed"] == 0)
disagree = sum(1 for r in all_rows if r["unique_answers"] > 1)
all_lens = sorted(L for r in all_rows for L in r["lengths"])
tot_fin = sum(r["finished"] for r in all_rows)

print("=" * 80)
print("BASELINE ACCURACY")
print("=" * 80)
print(f"  Pass@1 (avg over {total_samples} samples) : "
      f"{100*total_correct/total_samples:.1f}%")
print(f"  Paper reports for AIME25                : 91.4%")
print()
print("=" * 80)
print("PROBLEM BREAKDOWN  (this decides if our project has signal)")
print("=" * 80)
print(f"  Solved by ALL {K} samples   : {solved_all:2}/30   <- already perfect")
print(f"  Solved by SOME samples     : {solved_some:2}/30   <- *** OUR TARGET ***")
print(f"  Solved by NONE             : {solved_none:2}/30")
print(f"  Fully truncated (no answer): {truncated:2}/30")
print(f"  Samples disagreed          : {disagree:2}/30   <- where CLR can help")
print()
print("=" * 80)
print("LENGTHS")
print("=" * 80)
print(f"  Finished naturally : {tot_fin}/{total_samples} ({100*tot_fin/total_samples:.0f}%)")
print(f"  Median             : {all_lens[len(all_lens)//2]:,}")
print(f"  90th percentile    : {all_lens[int(len(all_lens)*0.9)]:,}")
print(f"  Longest            : {all_lens[-1]:,}")
print(f"  Total runtime      : {elapsed/60:.1f} min ({elapsed/3600:.2f} GPU-hours)")
print("=" * 80)

import csv
with open("outputs/phase5_baseline.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "truth", "correct", "K", "pass_at_1", "finished",
                "parsed", "unique_answers", "max_len", "predictions"])
    for r in all_rows:
        w.writerow([r["id"], r["truth"], r["correct"], K, r["pass_at_1"],
                    r["finished"], r["parsed"], r["unique_answers"],
                    max(r["lengths"]), r["predictions"]])
print("Saved: outputs/phase5_baseline.json  and  .csv")

In [1]:
# ============================================================================
# CELL 0 - SESSION SETUP
# Run this FIRST every time you open the notebook after a break.
# It is smart: if vLLM is already there, it skips the install instantly.
# ============================================================================

import importlib.util, os

os.makedirs("outputs", exist_ok=True)

if importlib.util.find_spec("vllm") is None:
    print("vLLM not found (fresh Kaggle session).")
    print("Installing now - takes 5-10 minutes. Ignore the red conflict warnings.")
    print()
    !pip install -q vllm
    print()
    print("=" * 60)
    print("INSTALL DONE.")
    print("=" * 60)
else:
    import vllm
    print(f"vLLM already installed (version {vllm.__version__}). Nothing to do.")

print()
print("Setup complete. You can run the next cell.")

vLLM already installed (version 0.26.0). Nothing to do.

Setup complete. You can run the next cell.


In [ ]:
# ============================================================================
# PHASE 6: BUDGET FORCING
# Generate once per budget, score twice (with / without forced termination).
# ============================================================================

import time, json, os, re, csv
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset

os.makedirs("outputs", exist_ok=True)
MODEL = "WeiboAI/VibeThinker-3B"
K = 4
BUDGETS = [4096, 8192]
FORCE_TOKENS = 256

# Phrase injected when the model runs out of budget
FORCE_PHRASE = (
    "\n\n</think>\n\n"
    "I have reasoned enough. Based on the work above, the final answer is \\boxed{"
)

ds = load_dataset("math-ai/aime25")["test"]
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# Small context => high concurrency => fast
llm = LLM(model=MODEL, dtype="float16", gpu_memory_utilization=0.90,
          max_model_len=max(BUDGETS) + FORCE_TOKENS + 1792,
          trust_remote_code=True)
print("\nModel loaded.\n")


def build_prompt(q):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)


def extract_boxed(text):
    idx = text.rfind("\\boxed{")
    if idx == -1:
        return None
    i, depth, out = idx + 7, 1, []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
        out.append(c); i += 1
    return "".join(out).strip()


def to_int(s):
    if s is None:
        return None
    m = re.search(r"-?\d+", s.replace(",", ""))
    return int(m.group()) if m else None


def first_int(s):
    """Parse the answer out of a forced continuation like '588} ...'"""
    m = re.search(r"-?\d+", s.replace(",", ""))
    return int(m.group()) if m else None


prompts = [build_prompt(ds[i]["problem"]) for i in range(30)]
truths = [to_int(str(ds[i]["answer"])) for i in range(30)]

results = {}
examples = []
t_start = time.time()

for BUDGET in BUDGETS:
    print("=" * 80)
    print(f"BUDGET = {BUDGET:,} tokens")
    print("=" * 80)

    # ---- Stage 1: normal generation, capped at BUDGET ----------------------
    p1 = SamplingParams(n=K, temperature=1.0, top_p=0.95, max_tokens=BUDGET)
    t0 = time.time()
    outs = llm.generate(prompts, p1)
    gen_time = time.time() - t0
    gen_tokens = sum(len(c.token_ids) for o in outs for c in o.outputs)
    print(f"  stage 1: {gen_time/60:.1f} min, {gen_tokens:,} tokens, "
          f"{gen_tokens/gen_time:.0f} tok/s\n")

    # ---- Score WITHOUT forcing (condition B) -------------------------------
    plain, trunc_jobs = [], []
    for pi, (o, truth) in enumerate(zip(outs, truths)):
        preds = []
        for ci, c in enumerate(o.outputs):
            pred = to_int(extract_boxed(c.text))
            preds.append(pred)
            if c.finish_reason != "stop":
                trunc_jobs.append((pi, ci, prompts[pi] + c.text + FORCE_PHRASE))
        plain.append(preds)

    b_correct = sum(1 for pi in range(30) for p in plain[pi] if p == truths[pi])
    n_trunc = len(trunc_jobs)
    print(f"  truncated samples: {n_trunc}/{30*K}")
    print(f"  [B] no forcing  : {100*b_correct/(30*K):.1f}%\n")

    # ---- Stage 2: force the truncated ones to commit (condition C) ---------
    forced = [row[:] for row in plain]
    force_time, force_tokens = 0.0, 0
    if trunc_jobs:
        print(f"  stage 2: forcing {n_trunc} truncated samples...")
        p2 = SamplingParams(n=1, temperature=0.0, max_tokens=FORCE_TOKENS)
        t0 = time.time()
        fouts = llm.generate([j[2] for j in trunc_jobs], p2)
        force_time = time.time() - t0
        force_tokens = sum(len(o.outputs[0].token_ids) for o in fouts)

        n_parsed = 0
        for (pi, ci, _), fo in zip(trunc_jobs, fouts):
            cont = fo.outputs[0].text
            val = first_int(cont)
            forced[pi][ci] = val
            if val is not None:
                n_parsed += 1
            if len(examples) < 4:
                examples.append({
                    "problem": pi, "truth": truths[pi],
                    "continuation": cont[:300], "parsed": val,
                })
        print(f"  stage 2: {force_time/60:.1f} min, "
              f"parsed {n_parsed}/{n_trunc} ({100*n_parsed/n_trunc:.0f}%)\n")

    c_correct = sum(1 for pi in range(30) for p in forced[pi] if p == truths[pi])
    print(f"  [C] WITH forcing: {100*c_correct/(30*K):.1f}%")
    print(f"  gain from forcing: {100*(c_correct-b_correct)/(30*K):+.1f} points\n")

    results[BUDGET] = {
        "budget": BUDGET,
        "truncated": n_trunc,
        "no_forcing_pct": round(100 * b_correct / (30 * K), 1),
        "forcing_pct": round(100 * c_correct / (30 * K), 1),
        "gain": round(100 * (c_correct - b_correct) / (30 * K), 1),
        "gen_tokens": gen_tokens,
        "force_tokens": force_tokens,
        "total_tokens": gen_tokens + force_tokens,
        "minutes": round((gen_time + force_time) / 60, 1),
        "plain_preds": plain,
        "forced_preds": forced,
    }

    with open("outputs/phase6_forcing.json", "w") as f:
        json.dump({"results": results, "examples": examples,
                   "truths": truths}, f, indent=2)
    print(f"  saved.\n")

# ---------------------------------------------------------------------------
BASE_PCT, BASE_TOKENS, BASE_HRS = 80.8, 4_600_000, 8.87

print("=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"{'Condition':<28}{'Acc':>8}{'Tokens':>14}{'GPU-hrs':>10}")
print("-" * 60)
print(f"{'A: 32k, no forcing':<28}{BASE_PCT:>7.1f}%{'~4.6M':>14}{BASE_HRS:>10.2f}")
for b, r in results.items():
    hrs = r["minutes"] / 60
    print(f"{'B: ' + f'{b//1024}k, no forcing':<28}"
          f"{r['no_forcing_pct']:>7.1f}%{r['gen_tokens']:>14,}{hrs:>10.2f}")
    print(f"{'C: ' + f'{b//1024}k, WITH forcing':<28}"
          f"{r['forcing_pct']:>7.1f}%{r['total_tokens']:>14,}{hrs:>10.2f}")
print("-" * 60)

best_b = max(results, key=lambda b: results[b]["forcing_pct"])
best = results[best_b]
print()
print("KEY COMPARISON")
print(f"  Baseline (32k, no forcing) : {BASE_PCT}%  in {BASE_HRS:.2f} GPU-hrs")
print(f"  Best forced ({best_b//1024}k + forcing) : {best['forcing_pct']}%  "
      f"in {best['minutes']/60:.2f} GPU-hrs")
print(f"  Accuracy delta  : {best['forcing_pct'] - BASE_PCT:+.1f} points")
print(f"  Compute speedup : {BASE_HRS/(best['minutes']/60):.1f}x cheaper")
print()

print("=" * 80)
print("SAMPLE FORCED CONTINUATIONS (sanity check - is it real or garbage?)")
print("=" * 80)
for e in examples:
    print(f"  problem {e['problem']}  truth={e['truth']}  parsed={e['parsed']}"
          f"  {'HIT' if e['parsed'] == e['truth'] else 'miss'}")
    print(f"    >>> {e['continuation'][:200]}")
    print()

with open("outputs/phase6_forcing.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["condition", "budget", "accuracy_pct", "truncated",
                "total_tokens", "gpu_hours"])
    w.writerow(["A_baseline", 32768, BASE_PCT, "", BASE_TOKENS, BASE_HRS])
    for b, r in results.items():
        w.writerow([f"B_noforce", b, r["no_forcing_pct"], r["truncated"],
                    r["gen_tokens"], round(r["minutes"] / 60, 2)])
        w.writerow([f"C_force", b, r["forcing_pct"], r["truncated"],
                    r["total_tokens"], round(r["minutes"] / 60, 2)])
print("=" * 80)
print(f"Total runtime: {(time.time()-t_start)/60:.1f} min")
print("Saved: outputs/phase6_forcing.json and .csv")

INFO 08-08 04:41:16 [api_utils.py:273] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 10240, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'WeiboAI/VibeThinker-3B'}
INFO 08-08 04:41:17 [model.py:623] Resolved architecture: Qwen2ForCausalLM
WARNING 08-08 04:41:17 [model.py:2123] Casting torch.bfloat16 to torch.float16.
INFO 08-08 04:41:17 [model.py:1788] Using max model len 10240
INFO 08-08 04:41:17 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-08 04:41:17 [vllm.py:1109] Asynchronous scheduling is enabled.
INFO 08-08 04:41:17 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=1259) INFO 08-08 04:41:21 [core.py:116] Initializing a V1 LLM engine (v0.26.0) with config: model='WeiboAI/VibeThinker-3B', speculative_config=None, tokenizer='WeiboAI/VibeThinker-3B', skip_tokenizer_init=False, t

[W808 04:41:23.373115298 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


(EngineCore pid=1259) INFO 08-08 04:41:24 [model_runner.py:284] Loading model from scratch...
(EngineCore pid=1259) ERROR 08-08 04:41:25 [fa_utils.py:253] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(EngineCore pid=1259) INFO 08-08 04:41:29 [cuda.py:482] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=1259) INFO 08-08 04:41:29 [weight_utils.py:869] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 5.75 GiB. Available RAM: 27.96 GiB.
(EngineCore pid=1259) INFO 08-08 04:41:29 [weight_utils.py:892] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=1259) INFO 08-08 04:41:35 [default_loader.py:430] Loading weights took 5.91 seconds
(EngineCore pid=1259) INFO 08-08 04:41:36 [model_runner.py:305] Model loading took 5.81 GiB and 12.438121 seconds
(EngineCore pid=1259) WARNING 08-08 04:41:36 [topk_topp_sampler.py:62] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=1259) INFO 08-08 04:41:41 [backends.py:1094] Using cache directory: /root/.cache/vllm/torch_compile_cache/4e28e6fb56/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=1259) INFO 08-08 04:41:41 [backends.py:1155] Dynamo bytecode transform time: 4.20 s
(EngineCore pid=1259) INFO 08-08 04:41:43 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.648 s
(EngineCore pid=1259) INFO 08-08 04:41:43 [decorators.py:311] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch